# WaSPS-DTW on CPAZMaL: SAR amplitude classification with Weibull distributions

This notebook demonstrates the full WaSPS-DTW pipeline on the CPAZMaL SAR dataset:

1. **Load the HDF5 dataset** and extract windowed time series.
2. **Estimate Weibull(k, λ) parameters** at each timestep using the method of log-cumulants.
3. **K-medoid classification**: compute per-class barycenters via SGD, then assign each test sample to the nearest barycenter using the Soft-DTW divergence with a Weibull local cost.
4. **Learning Shapelets classification**: train shapelets end-to-end with a Soft-DTW Wasserstein loss.
5. **Visualise** barycenters and classification results.

**Why Weibull?**  
SAR amplitude data (intensity raised to ½) follow a Weibull distribution whose shape parameter *k* encodes surface roughness and whose scale parameter *λ* encodes mean backscatter. Using a Weibull local cost for Soft-DTW directly compares time-series of *distributions* rather than individual pixel values, which is more robust to the high within-class spatial variability in SAR imagery.

> **Dataset**: CPAZMaL — PAZ/TerraSAR-X cryospheric SAR archive  
> **HuggingFace**: https://huggingface.co/datasets/musmb/CPAZMaL

In [ ]:
import sys
from pathlib import Path

# Make src/ importable
ROOT = Path().resolve().parent
SRC  = ROOT / 'src'
sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split

## 1. Load the dataset

Point `HDF5_PATH` at your local copy of `PAZTSX_CRYO_ML.hdf5`.
If the file is missing, `download_cpazmal()` will fetch it from HuggingFace.

In [ ]:
from dataloader import MLDatasetLoader, extract_time_series, download_cpazmal, estimate_weibull_params

HDF5_PATH = Path('/home/mgallet/Documents/Codes/Python/1_DONE'
                 '/CPAZMAL/DATASET/dataset_original/PAZTSX_CRYO_ML.hdf5')

if not HDF5_PATH.exists():
    print('Downloading CPAZMaL from HuggingFace…')
    HDF5_PATH = Path(download_cpazmal(str(HDF5_PATH.parent)))

loader = MLDatasetLoader(str(HDF5_PATH))

print(f'Classes: {loader.classes}')
print(f'Number of groups: {loader.n_groups}')

## 2. Extract windowed time series

We extract 12×12 pixel windows from the descending orbit, HH polarisation, amplitude-scaled images.
Each window is reshaped to `(T_train, W²)` where `T_train` is the number of acquisitions in Jan–Oct 2020 and `W²=144` is the number of pixels per window.  
Each row at time `t` holds 144 i.i.d. Weibull samples.

For a quick demo we limit to the first 10 groups.

In [ ]:
# Optional: limit to first N groups for speed
MAX_GROUPS = 10
orig_index = loader.class_index
trimmed, count = {}, 0
for cls, entries in orig_index.items():
    keep = entries[:max(1, MAX_GROUPS // max(len(orig_index), 1))]
    trimmed[cls] = keep
    count += len(keep)
    if count >= MAX_GROUPS:
        break
loader.class_index = trimmed

dataset = extract_time_series(
    loader=loader,
    window_size=12,
    orbit='DSC',
    polarization='HH',
    train_start='20200101', train_end='20201031',
    predict_start='20201101', predict_end='20201231',
    scale_type='amplitude',
    max_mask_value=1, max_mask_percentage=10.0, min_valid_percentage=50.0,
    skip_optim_offset=True,
    verbose=True,
)

X_raw        = dataset['X_train']   # list of (T_train, 144) arrays
y            = dataset['y']
idx_to_class = dataset['class_names']

print(f'\nSample shape: {X_raw[0].shape}  (T_train timesteps × 144 pixels)')
print(f'Total samples: {len(X_raw)}')

## 3. Estimate Weibull(k, λ) parameters

For each sample and each timestep, we fit a Weibull distribution to the 144 pixel values  
using the **method of log-cumulants**:

$$\hat{k} = \frac{\pi}{\sqrt{6}\, \sigma_{\log}}, \qquad \hat{\lambda} = \exp\!\left(\mu_{\log} - \frac{\psi(1)}{\hat{k}}\right)$$

where $\mu_{\log}$ and $\sigma_{\log}$ are the mean and standard deviation of $\log$(pixels).

In [ ]:
X_params = [estimate_weibull_params(s) for s in X_raw]

print(f'Parameter array shape per sample: {X_params[0].shape}')
print(f'  column 0 = k (shape),  column 1 = λ_scale')
print(f'  example: k={X_params[0][:3, 0].round(2)}, λ={X_params[0][:3, 1].round(3)}')

In [ ]:
# Visualise estimated k and λ for two classes
classes_present = sorted(set(y))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for cls in classes_present[:4]:
    idx_cls = np.where(y == cls)[0]
    k_mean   = np.mean([X_params[i][:, 0] for i in idx_cls], axis=0)
    lam_mean = np.mean([X_params[i][:, 1] for i in idx_cls], axis=0)
    axes[0].plot(k_mean,   label=idx_to_class[cls], alpha=0.8)
    axes[1].plot(lam_mean, label=idx_to_class[cls], alpha=0.8)

axes[0].set(title='Mean shape parameter k per class', xlabel='Time step', ylabel='k')
axes[1].set(title='Mean scale parameter λ per class', xlabel='Time step', ylabel='λ')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Train / test split

In [ ]:
idx_all = np.arange(len(X_raw))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.2, stratify=y, random_state=42)

X_tr = [X_params[i] for i in tr_idx]
X_te = [X_params[i] for i in te_idx]
y_tr = y[tr_idx]
y_te = y[te_idx]

print(f'Train: {len(y_tr)}  Test: {len(y_te)}')

## 5. K-medoid classification

We compute one **Weibull barycenter** per class via SGD minimising:

$$\bar{\theta} = \arg\min_b \sum_{x \in \text{class}} \mathrm{SDTW}_{\gamma}(b, x; W_2^2)$$

Then each test sample is assigned to the class with the smallest **Soft-DTW divergence**:

$$D_\gamma(x, b) = \mathrm{SDTW}(x,b) - \tfrac{1}{2}(\mathrm{SDTW}(x,x) + \mathrm{SDTW}(b,b))$$

In [ ]:
from sdtw.classification_methods import compute_barycenter_wasserstein_sgd, compute_sdtw_distance_weibull

GAMMA      = 10.0
SGD_EPOCHS = 20
SGD_LR     = 0.05

barycenters = {}
for cls in np.unique(y_tr):
    cls_samples = [X_tr[i] for i in range(len(X_tr)) if y_tr[i] == cls]
    print(f'  Class {idx_to_class[cls]:12s} — {len(cls_samples)} samples … ', end='')
    barycenters[cls] = compute_barycenter_wasserstein_sgd(
        cls_samples, gamma=GAMMA, learning_rate=SGD_LR,
        num_epochs=SGD_EPOCHS, batch_size=4, distribution='weibull', verbose=False,
    )
    print(f'barycenter shape: {barycenters[cls].shape}')

print('\nBarycenters computed.')

In [ ]:
# Visualise class barycenters (k and λ channels)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for cls, bary in barycenters.items():
    axes[0].plot(bary[:, 0], label=idx_to_class[cls])
    axes[1].plot(bary[:, 1], label=idx_to_class[cls])
axes[0].set(title='Barycenter: shape k', xlabel='Time step', ylabel='k')
axes[1].set(title='Barycenter: scale λ', xlabel='Time step', ylabel='λ')
for ax in axes:
    ax.legend(fontsize=8)
plt.suptitle('Per-class Weibull barycenters')
plt.tight_layout()
plt.show()

In [ ]:
# Classify by nearest barycenter
y_pred_kmedoid = []
for p in X_te:
    dists = {cls: compute_sdtw_distance_weibull(p, barycenters[cls], gamma=GAMMA, divergence=True)
             for cls in barycenters}
    y_pred_kmedoid.append(min(dists, key=dists.get))
y_pred_kmedoid = np.array(y_pred_kmedoid)

f1_w = f1_score(y_te, y_pred_kmedoid, average='weighted', zero_division=0)
f1_m = f1_score(y_te, y_pred_kmedoid, average='macro',    zero_division=0)
print(f'K-medoid — F1 weighted: {f1_w:.4f}  |  F1 macro: {f1_m:.4f}')

In [ ]:
# Confusion matrix
present = sorted(set(y_te))
names   = [idx_to_class[i] for i in present]
cm      = confusion_matrix(y_te, y_pred_kmedoid, labels=present)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('K-medoid Weibull — Confusion matrix')
plt.tight_layout()
plt.show()

## 6. Learning Shapelets classification

Shapelets are short reference sub-sequences trained end-to-end.  
Here we use a **Soft-DTW Wasserstein** distance between Weibull parameter sequences as the shapelet distance, making the shapelets directly comparable in distribution space.

In [ ]:
from experiments.shapelets_classifier import train_shapelets_classifier, predict_shapelets_classifier

T_train = X_tr[0].shape[0]
shapelet_len = max(3, T_train // 4)

clf, state = train_shapelets_classifier(
    train_samples=X_tr,
    y_train=y_tr,
    dist_measure='soft_dtw_wasserstein',
    epochs=10,
    batch_size=32,
    learning_rate=1e-3,
    shapelets_size_and_len={shapelet_len: 2},
    shapelets_gamma=GAMMA,
    shapelets_num_per_scale=2,
    seed=42,
    verbose=0,
)
print(f'Training time: {state["train_time"]:.1f}s')

In [ ]:
y_pred_shapelets = predict_shapelets_classifier(
    clf=clf, state=state, test_samples=X_te, batch_size=32)

f1_w = f1_score(y_te, y_pred_shapelets, average='weighted', zero_division=0)
f1_m = f1_score(y_te, y_pred_shapelets, average='macro',    zero_division=0)
print(f'Shapelets — F1 weighted: {f1_w:.4f}  |  F1 macro: {f1_m:.4f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_te, y_pred_shapelets, labels=present)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('Learning Shapelets Weibull — Confusion matrix')
plt.tight_layout()
plt.show()

## 7. Summary

| Method | F1 weighted | F1 macro |
|--------|------------|----------|
| K-medoid Weibull | — | — |
| Learning Shapelets Weibull | — | — |

*(Fill in after running the full dataset.  Here we used only a few groups for speed.)*